# **Preparation Notebook**



---
## Setup Environment

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd

from utstd.folders import *
from utstd.ipyrenders import *

at = AtFolder(
    course_code=36106,
    assignment="AT1",
)
at.run()

import warnings
warnings.simplefilter(action='ignore')

---
## Student Information

In [ ]:
student_name = "Nonthawat Praisompong"
student_id = "25233750"

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_name', value=student_name)

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h1", key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

> If you are using additional packages, you need to install them here using the command: `! pip install <package_name>`

### 0.b Import Packages

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import altair as alt
import scipy.stats as stats
from pathlib import Path

---
## A. Feature Selection


In [ ]:
# put the path folder into the folder_path by using Path
folder_path = Path("/content/gdrive/MyDrive/36106/assignment/AT1/data")

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Load training data
try:
  training_df = pd.read_csv(folder_path / "car_insurance_premium_training.csv")
  validation_df = pd.read_csv(folder_path / "car_insurance_premium_validation.csv")
  testing_df = pd.read_csv(folder_path / "car_insurance_premium_testing.csv")
except Exception as e:
  print(e)

### A.1 Approach 1

## There are 2 methods that I considered using for feature selection

*   Correlation Matrix (For selecting numeric features)
*   ANOVA (For selecting cateogry features)
** **  





**However There are some preparation that require which are to obtain the precise result**


*   Remove unrelated variables
*   Convert date columns
*   Correct the data type

These process are considered in data cleanning but to increase the precision feature this is mandatory.

** **


**Firstly, I will drop variables that I consider they are not related to my target variable (Net Premium Amout)**

The 12 variables that I am going to remove consist of

* customer_id
* prefix
* first_name
* last_name
* phone_number
* email
* secondary_address
* building_number
* street_name
* street_suffix
* suburb
* lapsed_date

Fortunately, prefix and lapsed_date are columns that contain most outliers.

In [ ]:
## To not manipulate the dataset, I have decided to create a new data frame.
training_df_ap = training_df


### select the variables that I am going to drop in the new variable
drop_columns = ['customer_id', 'prefix', 'first_name', 'last_name', 'phone_number', 'email',
                'secondary_address', 'building_number', 'street_name', 'street_suffix', 'suburb', 'lapsed_date']     ## to make it convenience, I have put into the list


## Following, drop these columns from this data frame.
training_df_ap = training_df_ap.drop(columns = drop_columns)

In [ ]:
## Then, checking the correctness
training_df_ap.info()  ## They have been removed

In [ ]:
# <Student to fill this section and then remove this comment>
feature_selection_1_insights = """
Removing unrelated variables helps in reducing the process of data preparation, which improves the data quality and readability. Additionally, reducing the noise and improving the quality of the data.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_1_insights', value=feature_selection_1_insights)

### A.2 Approach 2

The Datetime feature engineering and changing the dtype.
1. Changing the Dtype of the columns from object to datetime or category
2. Creating the new variable for 'birth_date', 'driving_license_date' and 'contract_start_date' by subtracting with current year (2025) to find the age or lenght of contract

In [ ]:
### Fistly, Change dtype to datatime
training_df_ap['birth_date'] = pd.to_datetime(training_df_ap['birth_date'], format='%Y-%m-%d')
training_df_ap['driving_license_date'] = pd.to_datetime(training_df_ap['driving_license_date'], format='%Y-%m-%d')
training_df_ap['contract_start_date'] = pd.to_datetime(training_df_ap['contract_start_date'], format='%Y-%m-%d')
training_df_ap['last_renewal_date'] = pd.to_datetime(training_df_ap['last_renewal_date'], format='%Y-%m-%d')
training_df_ap['next_renewal_date'] = pd.to_datetime(training_df_ap['next_renewal_date'], format='%Y-%m-%d')


### Secondly, I want to convert the year into age or how long by minus from the current year (2025)
## However, we have to extract only year out first into new features
training_df_ap['customer_age']        = 2025 - training_df_ap['birth_date'].dt.year.astype('int64')     ### .dt.year to extract only year
training_df_ap['driving_experience']  = 2025 - training_df_ap['driving_license_date'].dt.year.astype('int64')
training_df_ap['contract_years']      = 2025 - training_df_ap['contract_start_date'].dt.year.astype('int64')
training_df_ap['last_contact_date']   = 2025 - training_df_ap['last_renewal_date'].dt.year.astype('int64')
# This is the feature engineering process that converts indirectly related variables into related variables
# On the other hand, It creates the new feature that directly reflect customer characteristics such as customer age, car age and how many year that each customer drive a car.

### Then, we remove the old features
training_df_ap.drop(columns= ['birth_date', 'driving_license_date', 'contract_start_date'], inplace = True)


## Checking dtype
print(training_df_ap[['customer_age', 'driving_experience', 'contract_years']].info() )  ## Dtype has been change properly
print(' ')


### Then checking the values
v_check = ['customer_age', 'driving_experience', 'contract_years']

for col in v_check:
    print(f'What are thier unique value {col}: {training_df_ap[col].unique()}')
    print('  ')   ### The result has been display correctly.

In [ ]:
### Secondly, changing to category dtype for ANOVA testing
training_df_ap[['gender', 'distribution_channel', 'payment_method', 'policy_type', 'second_driver', 'vehicle_doors', 'vehicle_fuel_type']] = training_df_ap[['gender', 'distribution_channel', 'payment_method',
                                                                                                                                                             'policy_type', 'second_driver', 'vehicle_doors',
                                                                                                                                                             'vehicle_fuel_type']].astype('category')
training_df_ap[['gender', 'distribution_channel', 'payment_method', 'policy_type', 'second_driver', 'vehicle_doors', 'vehicle_fuel_type']].info()  ## Checking the correctness

In [ ]:
### Lastly, transfomr matriculation_year into cars's age which directly tell use about age of the cars in the dataset.
training_df_ap['car_age'] = 2025 - training_df_ap['matriculation_year'].astype(int)

### Then, we remove the old features
training_df_ap.drop(columns= ['matriculation_year'], inplace = True)

In [ ]:
# <Student to fill this section>
feature_selection_2_insights = """
Feature engineering transformed the time variable into meaningful variables, which might have potentially been useful for the analysis, such as age, driving experiences (year) and age of the car.
On the other hand, we transform ordinal value to numerical variables which are considered relating to our target variables.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_2_insights', value=feature_selection_2_insights)

### A.3 Approach "\<Correlation matrix illust the multicorlinearity\>"

> In this case, we are going to use correlation heatmap to visualise the relationship between numeric variables.

This is the primary approach for considering features in the model. The heatmap will display the correlation between each independent feature and net_premium_amount, showing the weight of the correlation. In Addition, the map show the multicollinearity between each independent variable which if it more than threshold or 0.65 then, we will remove it from our model.

In [ ]:
### Creating the correlation heatmap for feature selection, multicollinearity checking.

## creating the variables that contain only numeric values
numeric_col = training_df_ap.select_dtypes(include = 'number').columns

# Then create the heatmap to illustrate correlation
corr_matrix = training_df_ap[numeric_col].corr(numeric_only=True)
plt.figure(figsize=(18, 18))
sns.heatmap(corr_matrix, annot = True, fmt = ".2f",
            cmap ='coolwarm', square = True, cbar_kws = {'shrink': .8})
plt.title('Correlation Matrix')
plt.xticks(rotation = 45, ha = 'right')
plt.yticks(rotation = 0)
plt.tight_layout()
plt.show()

# In this process, we set correlation at 0.65 as a threshold for preveting multicollinearity.
# Overall, between target feature and each independent variable have weak correlation and some of has negative correlation toward target variable (customer_age, car_age, contract_year, driving_experience)
# As the result, max_policies_held, contract_year, vehicle_weight, vehicle_length, total_claims_number_ratio will be removed from the feature selection due to correlation is greater than 0.65.
# For total_claims_number_ratio, I decided to remove because it overlap with other related features within the same group.

In [ ]:
###  Creating One-way ANOVA for testing category features in for model selection.

from scipy.stats import f_oneway

anova_results = {}
category_col = training_df_ap.select_dtypes(include = 'category')

for c in category_col:
    groups = [group['net_premium_amount'] for _, group in training_df_ap.groupby(c)]
    if len(groups) > 1:
        f_stat, p_val = f_oneway(*groups)
        anova_results[c] = {'F-statistic': f_stat, 'p-value': p_val}

anova_df = pd.DataFrame(anova_results).T.sort_values(by='F-statistic', ascending=False)
print(anova_df.round(3))

# According to this, gender is the only feature that doesn't statistic significant.
# payment_method has the outstanding F-stat while vehicle_doors has the lowest.
# As a result, gender will be removed

In [ ]:
# <Student to fill this section>
feature_selection_n_insights = """
For selecting features into the model, I decide to use 3 criteria; domain knowledge, correlation matrix and ANOVA test. Firstly, I use my domain knowledge to consider how important of each features.
However, relying only on domain knowledge would create bias selection leading to create bias result for business decision making which might cause unexpected lost.
To minimise the bias, statistical testing was applied for feature selection. Correlation heatmap for checking multicollinearity in numeric features and One-way ANOVA for testing differences in mean and statistically significant by using p-value.
On top of that, Domain knowledge is still applied for considering which features we should remove in the correlation matrix.
In summary, these criteria reduce bias in the feature selection and this help our model contain only significant factors toward the prediction.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_n_insights', value=feature_selection_n_insights)

### A.z Final Selection of Features

In [ ]:
## There are 6 category features and 14 numeric features in our feature list. 20 features in total.

features_list = ['payment_method', 'vehicle_fuel_type', 'distribution_channel', 'second_driver', 'policy_type', 'vehicle_doors',
                 'seniority', 'current_policies_held', 'max_products_held', 'lapsed_policies', 'net_premium_amount', 'total_claims_number_in_current_year',
                 'total_claims_number_in_history', 'vehicle_horsepower', 'vehicle_cylinder', 'vehicle_value', 'birth_date', 'driving_license_date', 'last_renewal_date', 'matriculation_year']
## For these features " 'birth_date', 'driving_license_date', 'last_renewal_date', 'matriculation_year' ", we have to convert them again for each dataset (by subtracting 2025)

target_name = 'net_premium_amount'

In [ ]:
feature_selection_explanations = """
I have selected 20 features for training a machine learning model. These features have passed the statistical testing, which might be relevant to the net premium amount.
Plus they were considered by domain knowledge.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_selection_explanations', value=feature_selection_explanations)

---
## B. Data Cleaning

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets
try:
  training_df_clean = training_df[features_list].copy()
  validation_df_clean = validation_df[features_list].copy()
  testing_df_clean = testing_df[features_list].copy()
except Exception as e:
  print(e)

In [ ]:
training_df_clean.info()

### B.1 Fixing "\<Impute the missing values with mode\>"

In [ ]:
# Due to the feature with the missing value is vehicle_fuel_type (category column)
# Then, imputing with the mode is the most suitable.

training_impute = training_df_clean.select_dtypes(include = ['object']).columns
validation_impute = validation_df_clean.select_dtypes(include = ['object']).columns
testing_impute = testing_df_clean.select_dtypes(include = ['object']).columns

for col in training_impute:
    c_mode = training_df_clean[col].mode()[0]
    training_df_clean[col].fillna(c_mode, inplace=True)

for col in validation_impute:
    c_mode = validation_df_clean[col].mode()[0]
    validation_df_clean[col].fillna(c_mode, inplace=True)

for col in testing_impute:
    c_mode = testing_df_clean[col].mode()[0]
    testing_df_clean[col].fillna(c_mode, inplace=True)

In [ ]:
data_cleaning_1_explanations = """
The reason I decided to impute the missing values with the mode for category columns instead of removing them because it helps in minimising the data loss and ensures the dataset remains representative
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_1_explanations', value=data_cleaning_1_explanations)

### B.2 Fixing "\<the incorrect value\>"

In [ ]:
## Display the incorrect value (00/01/1900) in distribution channel
print(training_df_clean['distribution_channel'].value_counts())
print(" ")
print(validation_df_clean['distribution_channel'].value_counts())
print(" ")
print(testing_df_clean['distribution_channel'].value_counts())

In [ ]:
## In distribution_channel there are 00/01/1900 which is incorrect values.
# Then I decide to impute incorrect values with the mode

impute_col = ['distribution_channel']

for col in impute_col:
    c_mode = training_df_clean[col].mode()[0]
    training_df_clean[col].replace('00/01/1900', c_mode, inplace=True)

    c_mode = validation_df_clean[col].mode()[0]
    validation_df_clean[col].replace('00/01/1900', c_mode, inplace=True)

    c_mode = testing_df_clean[col].mode()[0]
    testing_df_clean[col].replace('00/01/1900', c_mode, inplace=True)

In [ ]:
## In horsepower, there is value = 0 which is incorrect then, I decide to remove it to maintain the data correctness.
# On top of that,

training_df_clean = training_df_clean.drop(training_df_clean[training_df_clean['vehicle_horsepower'] == 0].index)
validation_df_clean = validation_df_clean.drop(validation_df_clean[validation_df_clean['vehicle_horsepower'] == 0].index)
testing_df_clean = testing_df_clean.drop(testing_df_clean[testing_df_clean['vehicle_horsepower'] == 0].index)

In [ ]:
# <Student to fill this section and then remove this comment>
data_cleaning_2_explanations = """
To minimise the data lost, I decide to replace incorrect values from category features with mode. In contrast, I realise that impute with mode might cause bias in analysis.
However, from the propotion of the incorrect values with in each dataset, I considered, doing this might not cause too much bias into this anslysis.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_2_explanations', value=data_cleaning_2_explanations)

### B.3 Fixing "\<Data Types\>"

In [ ]:
### Changing the data types to minimise meaningless results, it help algorith to train the data properly

# To category data type
training_df_clean[['payment_method', 'vehicle_fuel_type', 'distribution_channel', 'second_driver', 'policy_type', 'vehicle_doors']] = training_df_clean[['payment_method', 'vehicle_fuel_type',
                                                                                                                                                      'distribution_channel', 'second_driver',
                                                                                                                                                      'policy_type', 'vehicle_doors']].astype('category')

validation_df_clean[['payment_method', 'vehicle_fuel_type', 'distribution_channel', 'second_driver', 'policy_type', 'vehicle_doors']] = validation_df_clean[['payment_method', 'vehicle_fuel_type',
                                                                                                                                                      'distribution_channel', 'second_driver',
                                                                                                                                                      'policy_type', 'vehicle_doors']].astype('category')

testing_df_clean[['payment_method', 'vehicle_fuel_type', 'distribution_channel', 'second_driver', 'policy_type', 'vehicle_doors']] = testing_df_clean[['payment_method', 'vehicle_fuel_type',
                                                                                                                                                      'distribution_channel', 'second_driver',
                                                                                                                                                      'policy_type', 'vehicle_doors']].astype('category')


# To int64 data type
training_df_clean['net_premium_amount'] = training_df_clean['net_premium_amount'].astype('int64')
validation_df_clean['net_premium_amount'] = validation_df_clean['net_premium_amount'].astype('int64')
testing_df_clean['net_premium_amount'] = testing_df_clean['net_premium_amount'].astype('int64')


# To datetime data type
training_df_clean['birth_date'] = pd.to_datetime(training_df_clean['birth_date'], format='%Y-%m-%d')
training_df_clean['driving_license_date'] = pd.to_datetime(training_df_clean['driving_license_date'], format='%Y-%m-%d')
training_df_clean['last_renewal_date'] = pd.to_datetime(training_df_clean['last_renewal_date'], format='%Y-%m-%d')

validation_df_clean['birth_date'] = pd.to_datetime(validation_df_clean['birth_date'], format='%Y-%m-%d')
validation_df_clean['driving_license_date'] = pd.to_datetime(validation_df_clean['driving_license_date'], format='%Y-%m-%d')
validation_df_clean['last_renewal_date'] = pd.to_datetime(validation_df_clean['last_renewal_date'], format='%Y-%m-%d')

testing_df_clean['birth_date'] = pd.to_datetime(testing_df_clean['birth_date'], format='%Y-%m-%d')
testing_df_clean['driving_license_date'] = pd.to_datetime(testing_df_clean['driving_license_date'], format='%Y-%m-%d')
testing_df_clean['last_renewal_date'] = pd.to_datetime(testing_df_clean['last_renewal_date'], format='%Y-%m-%d')

In [ ]:
# <Student to fill this section and then remove this comment>
data_cleaning_3_explanations = """
Fixing the datatype prevent the model or algorithm treat the feature incorrect causing incorrect result. Dealing with this issue ensure that the feature will be analyse correctly.
On top of that, it help in further process of data preparation which is one-hot-encoding (pd.get_dummies)
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_cleaning_3_explanations', value=data_cleaning_3_explanations)

---
## C. Feature Engineering

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  training_df_eng = training_df_clean.copy()
  validation_df_eng = validation_df_clean.copy()
  testing_df_eng = testing_df_clean.copy()
except Exception as e:
  print(e)

### C.1 New Feature "\<customer_age\>"



In [ ]:
## In this case, I create the new feature by transforming birth_date and subtracting 2025
training_df_eng['customer_age']  = 2025 - training_df_eng['birth_date'].dt.year.astype('int64')
validation_df_eng['customer_age']  = 2025 - validation_df_eng['birth_date'].dt.year.astype('int64')
testing_df_eng['customer_age']  = 2025 - testing_df_eng['birth_date'].dt.year.astype('int64')

In [ ]:
# <Student to fill this section and then remove this comment>
feature_engineering_1_explanations = """
This feature direclty reflect to customer characteristic which each higher might have a chance to pay for more insurance product and higher price package.
On the other hand, higher age are likely to have more income or buy own an expensive car which directly influence to increasing of net premium amount.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_1_explanations', value=feature_engineering_1_explanations)

### C.2 New Feature "\<driving_experience\>"



In [ ]:
## I created the driving experiences by using the year that each customer got their driving licence - 2025
training_df_eng['driving_experience']  = 2025 - training_df_eng['driving_license_date'].dt.year.astype('int64')
validation_df_eng['driving_experience']  = 2025 - validation_df_eng['driving_license_date'].dt.year.astype('int64')
testing_df_eng['driving_experience']  = 2025 - testing_df_eng['driving_license_date'].dt.year.astype('int64')

In [ ]:
# <Student to fill this section and then remove this comment>
feature_engineering_2_explanations = """
Driving experience determine the driving efficiency which likely influence the number of accident. There are many time net premium usually increase based on the number of accident.
I believe that higher experience tend to have lower net premium.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_2_explanations', value=feature_engineering_2_explanations)

### C.3 New Feature "\<car_age\>"



In [ ]:
## I created the age of the car feature by using matriculation_year - 2025 then we got the age of each car
training_df_eng['car_age'] = 2025 - training_df_eng['matriculation_year'].astype('int64')
validation_df_eng['car_age'] = 2025 - validation_df_eng['matriculation_year'].astype('int64')
testing_df_eng['car_age'] = 2025 - testing_df_eng['matriculation_year'].astype('int64')

In [ ]:
# <Student to fill this section and then remove this comment>
feature_engineering_3_explanations = """
New car might has higher new premium amount. I believe, If the car are close to the current year 2025, it tend to has higher net premium.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_3_explanations', value=feature_engineering_3_explanations)

### C.4 New feature "\<last_contact_date\>"



In [ ]:
## The last new feature that I added to this dataset is last_contact_date. This indicates how long they have been made a contract with.
training_df_eng['last_contact_date'] = 2025 - training_df_eng['last_renewal_date'].dt.year.astype('int64')
validation_df_eng['last_contact_date'] = 2025 - validation_df_eng['last_renewal_date'].dt.year.astype('int64')
testing_df_eng['last_contact_date'] = 2025 - testing_df_eng['last_renewal_date'].dt.year.astype('int64')

In [ ]:
## After, finishing all feature engineering, We have to remove the old feature out.
training_df_eng.drop(columns = ['birth_date', 'driving_license_date', 'last_renewal_date', 'matriculation_year'], inplace = True)
validation_df_eng.drop(columns = ['birth_date', 'driving_license_date', 'last_renewal_date', 'matriculation_year'], inplace = True)
testing_df_eng.drop(columns = ['birth_date', 'driving_license_date', 'last_renewal_date', 'matriculation_year'], inplace = True)

In [ ]:
# <Student to fill this section and then remove this comment>
feature_engineering_n_explanations = """
For this fearue, it is indicate, how long that we made a latest contract with the company which I believer lower might indicate the frequency of the continue contract.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='feature_engineering_n_explanations', value=feature_engineering_n_explanations)

### B.4 Fixing "\<Removing Outlier\>"

In [ ]:
### REMOVING The outliers
## Creating a function for removing outliers.
# Due to removing outliers from all numeric features, create huge data is lost.
# However, I decide to removed outliers from some columns that I considered it is sensitive to outlier out.
# From my trials, removing outliers from seniority created a huge data loss. To maintain the data, I decide to leave it.

rm_columns = ['customer_age', 'driving_experience', 'car_age', 'vehicle_value', 'vehicle_cylinder', 'vehicle_horsepower', 'current_policies_held']

def remove_outliers(df):
    df_clean = df.copy()
    for col in df_clean[rm_columns].columns:
        Q1 = np.percentile(df_clean[col], 25)
        Q3 = np.percentile(df_clean[col], 75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        df_clean = df_clean[(df_clean[col] >= lower_bound) & (df_clean[col] <= upper_bound)]
    return df_clean

# For training dataset
print("Training Before:", training_df_eng.shape)
training_df_eng = remove_outliers(training_df_eng)
print("Training After:", training_df_eng.shape)

# For validation dataset
print("Validation Before:", validation_df_eng.shape)
validation_df_eng = remove_outliers(validation_df_eng)
print("Validation:", validation_df_eng.shape)

# For testing dataset
print("Testing Before:", testing_df_eng.shape)
testing_df_eng = remove_outliers(testing_df_eng)
print("Testing After:", testing_df_eng.shape)

---
## D. Data Preparation for Modeling

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
# Create copy of datasets

try:
  X_train = training_df_eng.copy()
  X_val = validation_df_eng.copy()
  X_test = testing_df_eng.copy()

  y_train = X_train.pop(target_name)
  y_val = X_val.pop(target_name)
  y_test = X_test.pop(target_name)
except Exception as e:
  print(e)

### D.1 Data Transformation (Log transformation)

In [ ]:
## Applying log transformation for normalising car characteristic features and some feature that sensitive to the outlier.
log_feature = ['vehicle_horsepower', 'vehicle_cylinder', 'vehicle_value']

# Apply log transformation to the specified columns in X_train, X_val, and X_test

for c in log_feature:
    X_train[c] = np.log(X_train[c])
    X_val[c] = np.log(X_val[c])
    X_test[c] = np.log(X_test[c])

In [ ]:
X_train.describe().round()

In [ ]:
# <Student to fill this section and then remove this comment>
data_transformation_1_explanations = """
Removing outlier reduce the effect of the extreme value which make training process more stable by doing this, I expected that this will improves the performance of linear models (it help model fit better)
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_1_explanations', value=data_transformation_1_explanations)

### D.2 Data Transformation (One-hot encoding)

In [ ]:
## Applying one hot encoding for category variable.
cat_feature_train = X_train.select_dtypes(include = 'category').columns
cat_feature_validation = X_val.select_dtypes(include = 'category').columns
cat_feature_testing = X_test.select_dtypes(include = 'category').columns

## Creating dummy variables for each cat features
train_dummy = pd.get_dummies(X_train[cat_feature_train])
validation_dummy = pd.get_dummies(X_val[cat_feature_validation])
test_dummy = pd.get_dummies(X_test[cat_feature_testing])


In [ ]:
# Then connecting 2 tables together.
# But firstly, I have to drop category featurs from the data frame first

X_train.drop(columns = cat_feature_train , inplace = True)
X_val.drop(columns = cat_feature_validation , inplace = True)
X_test.drop(columns = cat_feature_testing , inplace = True)

# After that, let concatenate 2 sets together.
X_train = pd.concat([X_train, train_dummy], axis=1)
X_val = pd.concat([X_val, validation_dummy], axis=1)
X_test = pd.concat([X_test, test_dummy], axis=1)

In [ ]:
## Checking the shape of each set
print(X_train.shape)
print(" ")
print(X_val.shape)
print(" ")
print(X_test.shape)
# It seem like there is one columns missing which is policy_type_4 that doesn't exist in data train columns

In [ ]:
## We have to add policy_type_4 into and impute with 0 which indicate that there is no existing in training set.

# Create the new feature
X_train['policy_type_4'] = 0

# Then align validation and test set index for matching with the train set.
X_val = X_val[X_train.columns]
X_test = X_test[X_train.columns]

In [ ]:
# <Student to fill this section and then remove this comment>
data_transformation_2_explanations = """
This transform the category variables which the algorithm cannot handle into binary which model can use these categorical variables which might influence outcome for thier training which I expected it will improve the model performance.
In summary, it transforms category variables into a set that the set that algorithm can learn from.
"""

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL
print_tile(size="h3", key='data_transformation_2_explanations', value=data_transformation_2_explanations)

---
## E. Save Datasets

> Do not change this code

In [ ]:
# DO NOT MODIFY THE CODE IN THIS CELL

try:
  X_train.to_csv(folder_path / 'X_train.csv', index=False)
  y_train.to_csv(folder_path / 'y_train.csv', index=False)

  X_val.to_csv(folder_path / 'X_val.csv', index=False)
  y_val.to_csv(folder_path / 'y_val.csv', index=False)

  X_test.to_csv(folder_path / 'X_test.csv', index=False)
  y_test.to_csv(folder_path / 'y_test.csv', index=False)
except Exception as e:
  print(e)